In [ ]:
#Spatial Interpolation
import numpy as np
from scipy.interpolate import griddata

# File paths for loading data
cap_file_path = r"Your Path\Support_files\4_cap_coor_mri_scaled.txt"
grid_file_path = r"Your Path\Support_files\6_coor_vel_GOI.txt"

# Load the arrays (tab-separated)
Coor_Cap = np.loadtxt(cap_file_path, delimiter="\t", skiprows=1)  # Cap Data
Coor_Grid = np.loadtxt(grid_file_path, delimiter="\t", skiprows=1)  # Grid Data

# Extract x, y, z coordinates
x_grid, y_grid, z_grid = Coor_Grid[:, 0], Coor_Grid[:, 1], Coor_Grid[:, 2]  # Grid (Sparse Data)
x_fine, y_fine, z_fine = Coor_Cap[:, 0], Coor_Cap[:, 1], Coor_Cap[:, 2]  # Cap (Fine Data)

# Extract velocity components
v_x_grid, v_y_grid, v_z_grid = Coor_Grid[:, 3], Coor_Grid[:, 4], Coor_Grid[:, 5]
v_x_fine, v_y_fine, v_z_fine = Coor_Cap[:, 3], Coor_Cap[:, 4], Coor_Cap[:, 5]  # Initial values

# Find indices where velocity components are zero in the cap data
zero_indices = (v_x_fine == 0) & (v_y_fine == 0) & (v_z_fine == 0)

# Prepare data for interpolation
points_grid = np.vstack((x_grid, y_grid, z_grid)).T
points_fine = np.vstack((x_fine, y_fine, z_fine)).T

# Interpolate only at zero-velocity locations
v_x_fine[zero_indices] = griddata(points_grid, v_x_grid, points_fine[zero_indices], method='linear')
v_y_fine[zero_indices] = griddata(points_grid, v_y_grid, points_fine[zero_indices], method='linear')
v_z_fine[zero_indices] = griddata(points_grid, v_z_grid, points_fine[zero_indices], method='linear')

# Handle NaNs with nearest-neighbor interpolation
nan_indices = np.isnan(v_x_fine)
v_x_fine[nan_indices] = griddata(points_grid, v_x_grid, points_fine[nan_indices], method='nearest')

nan_indices = np.isnan(v_y_fine)
v_y_fine[nan_indices] = griddata(points_grid, v_y_grid, points_fine[nan_indices], method='nearest')

nan_indices = np.isnan(v_z_fine)
v_z_fine[nan_indices] = griddata(points_grid, v_z_grid, points_fine[nan_indices], method='nearest')

# Create the final interpolated array
Coor_Cap_int = np.hstack((Coor_Cap[:, :3], v_x_fine[:, np.newaxis], v_y_fine[:, np.newaxis], v_z_fine[:, np.newaxis]))

# Print results
print("\nCoor_Cap_int (with interpolated velocities):")
print(Coor_Cap_int)

# Save the new interpolated array (tab-separated)
output_file = r"Your Path\Support_files\9_Cap_interpolated.txt"
np.savetxt(output_file, Coor_Cap_int, delimiter="\t", header="x\ty\tz\tu\tv\tw", comments='', fmt='%.6f')

print(f"\nInterpolated cap data saved to: {output_file}")
print("Shape of Coor_Cap_int:", Coor_Cap_int.shape)

In [ ]:
#Cap 3D visualisation 
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# File path to the saved data
file_path = r"Your Path\Support_files\9_Cap_interpolated.txt"

# Load the data from the file, skipping the header
data = np.loadtxt(file_path, delimiter="\t", skiprows=1)

# Extract the first three columns (x, y, z coordinates)
x = data[:, 0]
y = data[:, 1]
z = data[:, 2]

# Create a 3D plot to display the points
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot the coordinates
ax.scatter(x, y, z, c='b', marker='o', s=2)  # Blue points with small size

# Labels for the axes
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# Set title
ax.set_title('3D Plot of Points from Coor_Cap_interpolated.txt')

# Show the plot
plt.show()

In [ ]:
#Optional: VTP file creation for velocity distribution check (Recommended)
import numpy as np
import pyvista as pv

# File paths for input and output
input_file_path = r"Your Path\Support_files\9_Cap_interpolated.txt"
output_vtp_path = r"Your Path\Support_files\Cap_interpolated_surface.vtp"

# Load the interpolated data
Coor_Cap_int = np.loadtxt(input_file_path, delimiter="\t", skiprows=1)

# Extract spatial coordinates and velocity components
x, y, z = Coor_Cap_int[:, 0], Coor_Cap_int[:, 1], Coor_Cap_int[:, 2]
u, v, w = Coor_Cap_int[:, 3], Coor_Cap_int[:, 4], Coor_Cap_int[:, 5]

# Create a PyVista point cloud
points = np.vstack((x, y, z)).T  # Combine x, y, z into an array of points
point_cloud = pv.PolyData(points)

# Perform Delaunay triangulation to generate a surface
surface = point_cloud.delaunay_2d()

# Add each velocity component as a separate point data array
surface["Velocity_U"] = u  # x-component of velocity
surface["Velocity_V"] = v  # y-component of velocity
surface["Velocity_W"] = w  # z-component of velocity

# Add the velocity magnitude as a separate scalar field
velocity_magnitude = np.sqrt(u**2 + v**2 + w**2)
surface["Velocity Magnitude"] = velocity_magnitude

# Save the surface as a VTP file
surface.save(output_vtp_path)

# Print a confirmation message
print(f"VTP surface file has been saved to: {output_vtp_path}")

# Optional: Visualize the surface
plotter = pv.Plotter()
plotter.add_mesh(surface, scalars="Velocity Magnitude", cmap="viridis")
plotter.add_axes()
plotter.show()

In [ ]:
#Spatial interpolation (all timesteps)
import numpy as np
import os
from scipy.interpolate import griddata
import re

# File paths
cap_file_path = r"Your Path\Support_files\4_cap_coor_mri_scaled.txt"
grid_folder_path = r"Your Path\Support_files\8_coor_vel_grid_time"
output_file_path = r"Your Path\Support_files\10_Coor_Cap_interpolated_timesteps.txt"

# Load Cap coordinates (fine data)
Coor_Cap = np.loadtxt(cap_file_path, delimiter="\t", skiprows=1)  # Skip header

# Extract x, y, z from Cap
x_fine, y_fine, z_fine = Coor_Cap[:, 0], Coor_Cap[:, 1], Coor_Cap[:, 2]
points_fine = np.vstack((x_fine, y_fine, z_fine)).T  # Fine grid points

# List and sort timestep files numerically
grid_files = sorted(
    [f for f in os.listdir(grid_folder_path) if f.startswith("coor_vel_grid_t")],
    key=lambda x: int(re.search(r"(\d+)", x).group())  # Extract numeric timestep
)

# Ensure correct number of timesteps
assert len(grid_files) == 24, f"Expected 24 timesteps, but found {len(grid_files)}."

# Initialize list for storing interpolated data
interpolated_data = []

# Process each timestep file
for t, grid_file in enumerate(grid_files):
    print(f"Processing timestep {t} from file {grid_file}...")

    # Load the sparse grid data for the current timestep
    grid_file_path = os.path.join(grid_folder_path, grid_file)
    Coor_ROI = np.loadtxt(grid_file_path, delimiter="\t", skiprows=1)  # Skip header

    # Extract sparse coordinates and velocity components
    x_sparse, y_sparse, z_sparse = Coor_ROI[:, 0], Coor_ROI[:, 1], Coor_ROI[:, 2]
    v_x_sparse, v_y_sparse, v_z_sparse = Coor_ROI[:, 3], Coor_ROI[:, 4], Coor_ROI[:, 5]

    points_sparse = np.vstack((x_sparse, y_sparse, z_sparse)).T  # Sparse grid points

    # Interpolate velocities onto fine grid
    v_x_fine = griddata(points_sparse, v_x_sparse, points_fine, method='linear')
    v_y_fine = griddata(points_sparse, v_y_sparse, points_fine, method='linear')
    v_z_fine = griddata(points_sparse, v_z_sparse, points_fine, method='linear')

    # Handle NaNs with nearest-neighbor interpolation
    v_x_fine[np.isnan(v_x_fine)] = griddata(points_sparse, v_x_sparse, points_fine[np.isnan(v_x_fine)], method='nearest')
    v_y_fine[np.isnan(v_y_fine)] = griddata(points_sparse, v_y_sparse, points_fine[np.isnan(v_y_fine)], method='nearest')
    v_z_fine[np.isnan(v_z_fine)] = griddata(points_sparse, v_z_sparse, points_fine[np.isnan(v_z_fine)], method='nearest')

    # Combine the interpolated data
    Coor_Cap_int_timestep = np.hstack((Coor_Cap[:, :3], v_x_fine[:, np.newaxis], v_y_fine[:, np.newaxis], v_z_fine[:, np.newaxis]))
    
    # Append the timestep data
    interpolated_data.append((t, Coor_Cap_int_timestep))

# Save the interpolated data to a single output file
with open(output_file_path, 'w') as f:
    for t, data in interpolated_data:
        f.write(f"aligned_comp_{t}:\n")
        f.write("x, y, z, u, v, w\n")
        np.savetxt(f, data, delimiter=",", fmt="%.6f")
        f.write("\n")  # Blank line for separation

print("\nAll timesteps processed and saved successfully.")